In [ ]:
%%capture
!pip install pyspark
!pip install findspark
!pip install pyngrok

In [ ]:
import getpass
import findspark
import pandas as pd
from pyspark.sql import types
from pyngrok import ngrok, conf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, LongType, StringType

findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName('testColab') \
        .getOrCreate()

In [ ]:

print("Enter your authtoken, which can be copied "
"from https://dashboard.ngrok.com/get-started/your-authtoken")
conf.get_default().auth_token = getpass.getpass()

ui_port = 4040
public_url = ngrok.connect(ui_port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{ui_port}\"")

Enter your authtoken, which can be copied from https://dashboard.ngrok.com/get-started/your-authtoken
··········


 * ngrok tunnel "https://postelementary-unactivated-zola.ngrok-free.dev" -> "http://127.0.0.1:4040"


In [ ]:
# 1. Force Spark to stay on 4040
spark = SparkSession.builder \
    .appName('testColab') \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

# 2. Check where Spark ACTUALLY landed
actual_ui_url = spark.sparkContext.uiWebUrl
print(f"Internal Spark UI is at: {actual_ui_url}")

# 3. Connect ngrok to that specific port
# (Using 4040 here because we forced it above)
ui_port = 4040
public_url = ngrok.connect(ui_port).public_url
print(f" * Public Spark UI: {public_url}")

Internal Spark UI is at: http://807c88733b53:4040
 * Public Spark UI: https://postelementary-unactivated-zola.ngrok-free.dev


In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

--2026-04-14 11:39:25--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.97.61, 52.85.97.194, 52.85.97.44, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.97.61|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 308924937 (295M) [application/x-www-form-urlencoded]
Saving to: ‘fhvhv_tripdata_2021-01.parquet’

fhvhv_tripdata_2021 100%[===================>] 294.61M   200MB/s    in 1.5s    

2026-04-14 11:39:27 (200 MB/s) - ‘fhvhv_tripdata_2021-01.parquet’ saved [308924937/308924937]



#Pyspark

In [ ]:
# Read the dataset using spark and view first 5 rows
df = spark.read.option("header", "true").parquet("fhvhv_tripdata_2021-01.parquet")
df.show(5)

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [ ]:
df.head(1)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 28, 9), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 31, 42), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, trip_miles=5.26, trip_time=923, base_passenger_fare=22.28, tolls=0.0, bcf=0.67, sales_tax=1.98, congestion_surcharge=2.75, airport_fee=None, tips=0.0, driver_pay=14.99, shared_request_flag='N', shared_match_flag='N', access_a_ride_flag=' ', wav_request_flag='N', wav_match_flag='N')]

In [ ]:
# Limit data to 500 rows and convert to a parquet

df.limit(500).write.mode('overwrite').parquet("trips_data.parquet")

In [ ]:
df_pandas = pd.read_parquet("trips_data.parquet")
df_pandas.head(2)

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B02682,B02682,2021-01-01 00:28:09,2021-01-01 00:31:42,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,5.26,...,1.98,2.75,NaN,0.0,14.99,N,N,,N,N
1,HV0003,B02682,B02682,2021-01-01 00:45:56,2021-01-01 00:55:19,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,3.65,...,1.63,0.00,NaN,0.0,17.06,N,N,,N,N


In [ ]:
df_pandas.shape

(500, 24)

In [ ]:
# Using python see datatypes

df_pandas.dtypes

,0
hvfhs_license_num,object
dispatching_base_num,object
originating_base_num,object
request_datetime,datetime64[us]
on_scene_datetime,datetime64[us]
pickup_datetime,datetime64[us]
dropoff_datetime,datetime64[us]
PULocationID,int64
DOLocationID,int64
trip_miles,float64


In [ ]:
# Convert data back to a spark dataframe

trips = spark.createDataFrame(df_pandas)
trips.head(2)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 28, 9), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 31, 42), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, trip_miles=5.26, trip_time=923, base_passenger_fare=22.28, tolls=0.0, bcf=0.67, sales_tax=1.98, congestion_surcharge=2.75, airport_fee=nan, tips=0.0, driver_pay=14.99, shared_request_flag='N', shared_match_flag='N', access_a_ride_flag=' ', wav_request_flag='N', wav_match_flag='N'),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 45, 56), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOL

In [ ]:
trips = trips.repartition(10)

In [ ]:
# You can see the schema of spart dtaframes

trips.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampType(), True), StructField('on_scene_datetime', TimestampType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_request_flag',

In [ ]:
trips.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp (nullable = true)
 |-- on_scene_datetime: timestamp (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_flag: string (nul

In [ ]:
# Selecting a few columns

trips.select('hvfhs_license_num', 'trip_time', 'pickup_datetime')

DataFrame[hvfhs_license_num: string, trip_time: bigint, pickup_datetime: timestamp]

In [ ]:
# filtering

pu_select = trips.filter(trips.PULocationID == '230')
pu_select.show(2)

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [ ]:
# to_date - convert datetime to date

trips.withColumn('pickup_date', F.to_date(trips.pickup_datetime)) \
     .withColumn('dropoff_date', F.to_date(trips.dropoff_datetime)) \
     .select('pickup_date', 'dropoff_date', 'DOLocationID') \
     .show(2)


+-----------+------------+------------+
|pickup_date|dropoff_date|DOLocationID|
+-----------+------------+------------+
| 2021-01-01|  2021-01-01|         203|
| 2021-01-01|  2021-01-01|         213|
+-----------+------------+------------+
only showing top 2 rows


In [ ]:
# UDFs - are python functions that can be used to define business rules and implement in pyspark

def create_base_id(base_num):
  num = int(base_num[1:])
  if num % 7 == 0:
    return f's/{num:03x}'
  elif num % 3 == 0:
    return f'a/{num:03x}'
  else:
    return f'e/{num:03x}'


In [ ]:
# apply UDFs n/b: all are transformations except show, which id an action

create_base_udf = F.udf(create_base_id, returnType=types.StringType())


trips.withColumn('pickup_date', F.to_date(trips.pickup_datetime)) \
     .withColumn('dropoff_date', F.to_date(trips.dropoff_datetime)) \
     .withColumn('base_id', create_base_udf(trips.dispatching_base_num)) \
     .select('pickup_date', 'dropoff_date', 'DOLocationID', 'base_id') \
     .show(2)


+-----------+------------+------------+-------+
|pickup_date|dropoff_date|DOLocationID|base_id|
+-----------+------------+------------+-------+
| 2021-01-01|  2021-01-01|          29|  e/9ce|
| 2021-01-01|  2021-01-01|         255|  e/9ce|
+-----------+------------+------------+-------+
only showing top 2 rows


In [ ]:
# Rename columns

trips.withColumnRenamed('dispatching_base_num', 'base_num') \
      .withColumnRenamed('on_scene_datetime', 'scene_datetime') \
      .select('base_num', 'scene_datetime').show(2)

+--------+-------------------+
|base_num|     scene_datetime|
+--------+-------------------+
|  B02764|2021-01-01 00:42:37|
|  B02617|2020-12-31 23:59:47|
+--------+-------------------+
only showing top 2 rows


In [ ]:
# Groupby

trips.groupBy('')

# Spark SQL

In [ ]:
# To use Spark sql, we need to tell spark that the dataframe should be a table >> create temp table

trips.createOrReplaceTempView('trips_data')

In [ ]:
spark.sql("""
select * from trips_data
 """).show(1)

+-----------------+--------------------+--------------------+-------------------+-----------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-----------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+---

In [ ]:
spark.sql("""
select sum(sales_tax) from trips_data
 """).show()

+-----------------+
|   sum(sales_tax)|
+-----------------+
|739.1500000000001|
+-----------------+

